# MI vs Magnitude — automated comparison
Runs prune → finetune → sample → metrics for each method, then prints one table.
SSIM (vs the original model, same seed) is the primary metric; FID is optional.

## Setup

In [1]:
#!git clone --branch mipp https://github.com/elliotcanter11/Diff-Pruning.git

Cloning into 'Diff-Pruning'...
remote: Enumerating objects: 1155, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 1155 (delta 127), reused 113 (delta 93), pack-reused 1001 (from 1)
Receiving objects: 100% (1155/1155), 31.58 MiB | 15.18 MiB/s, done.
Resolving deltas: 100% (582/582), done.


In [2]:
!git clone https://github.com/elliotcanter11/Diff-Pruning.git
!git -C Diff-Pruning checkout 7bee44e91ed8ee01865dac16ae5082a97b4bf5aa

fatal: destination path 'Diff-Pruning' already exists and is not an empty directory.
Note: switching to '7bee44e91ed8ee01865dac16ae5082a97b4bf5aa'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 7bee44e fixed MI, fixed notebook


In [3]:
%cd Diff-Pruning/

/content/Diff-Pruning


In [4]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 1.7 MB/s eta 0:00:00


In [5]:
!pip install -q pytorch-msssim

In [6]:
!python tools/extract_cifar10_hug.py --output data

README.md: 100% 5.16k/5.16k [00:00<00:00, 13.6MB/s]

plain_text/train-00000-of-00001.parquet: downloading bytes:  96% 115M/120M [00:02<00:00, 73.2MB/s, 9.90MB/s  ]
plain_text/train-00000-of-00001.parquet: downloading bytes: 100% 117M/117M [00:03<00:00, 37.0MB/s, 10.3MB/s  ]
plain_text/train-00000-of-00001.parquet: reconstructing file: 100% 120M/120M [00:03<00:00, 38.0MB/s, 11.1MB/s  ]

plain_text/test-00000-of-00001.parquet: downloading bytes:  88% 21.1M/23.9M [00:02<00:00, 34.4MB/s, 1.46MB/s  ]
plain_text/test-00000-of-00001.parquet: downloading bytes: 100% 23.3M/23.3M [00:02<00:00, 10.1MB/s, 2.17MB/s  ]
plain_text/test-00000-of-00001.parquet: reconstructing file: 100% 23.9M/23.9M [00:02<00:00, 10.4MB/s, 2.26MB/s  ]
Generating train split: 100% 50000/50000 [00:00<00:00, 194252.51 examples/s]
Generating test split: 100% 10000/10000 [00:00<00:00, 226352.08 examples/s]
100% 50000/50000 [00:29<00:00, 1692.10it/s]


In [7]:
!bash tools/convert_cifar10_ddpm_ema.sh

--2026-08-30 22:52:02--  https://heibox.uni-heidelberg.de/f/2e4f01e2d9ee49bab1d5/?dl=1
Resolving heibox.uni-heidelberg.de (heibox.uni-heidelberg.de)... 129.206.7.113
Connecting to heibox.uni-heidelberg.de (heibox.uni-heidelberg.de)|129.206.7.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://heibox.uni-heidelberg.de/seafhttp/files/64612568-6642-47e4-8ccc-1dfb54ce9844/model-790000.ckpt [following]
--2026-08-30 22:52:03--  https://heibox.uni-heidelberg.de/seafhttp/files/64612568-6642-47e4-8ccc-1dfb54ce9844/model-790000.ckpt
Reusing existing connection to heibox.uni-heidelberg.de:443.
HTTP request sent, awaiting response... 200 OK
Length: 143046049 (136M) [application/octet-stream]
Saving to: ‘pretrained/cifar10-ema-model-790000.ckpt’

pretrained/cifar10- 100%[===================>] 136.42M  2.73MB/s    in 55s     

2026-08-30 22:52:59 (2.49 MB/s) - ‘pretrained/cifar10-ema-model-790000.ckpt’ saved [143046049/143046049]

odict_keys(['temb.dense.0.weig

In [8]:
# only needed if COMPUTE_FID = True below (mkdir so np.savez has a dir to write into)
!mkdir -p run && python fid_score.py --save-stats data/cifar10_images run/fid_stats_cifar10.npz --device cuda:0 --batch-size 256

Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth
100% 91.2M/91.2M [00:06<00:00, 13.9MB/s]
Saving statistics for data/cifar10_images
Found 50000 files.
100% 196/196 [01:25<00:00,  2.30it/s]


## Config + run
SSIM is enough to compare criteria; leave `COMPUTE_FID = False` for fast iteration (samples only ~1k images instead of 10k). Turn it on for a final number.

The table reports, per method: params/MACs after pruning (sanity that greedy hit the ratio), **SSIM** (higher = closer to the original model), and for the MI methods the **overlap vs magnitude** (Jaccard of the pruned set + Spearman) — low overlap = MI is genuinely choosing different filters.

In [9]:
import os, subprocess, re

RATIO        = 0.3     # pruning ratio (0.5 stresses harder but noisier)
ITERS        = 1000    # finetuning steps (finetune seed is fixed, so runs are comparable)
WORKERS      = 12
SSIM_SAMPLES = 1000    # paired vs the original model; cheap + low-noise
COMPUTE_FID  = True   # SSIM alone is enough to iterate; True for a final FID
FID_SAMPLES  = 10000
DDIM_STEPS   = 100     # 50 ~halves sampling time; fair since all methods match
SAMPLE_BS    = 256     # raise if GPU VRAM is free (check nvidia-smi) -> faster sampling, no downside

# prune_ddpm_cifar10_mi.sh args: ratio  w_output  w_adjacency
METHODS = {
    'magnitude'   : f'bash scripts/prune_ddpm_cifar10.sh {RATIO}',
    #'mi_adjacency': f'bash scripts/prune_ddpm_cifar10_mi.sh {RATIO} 0.0 1.0',
    #'mi_output'   : f'bash scripts/prune_ddpm_cifar10_mi.sh {RATIO} 1.0 0.0',
    'mi_combined' : f'bash scripts/prune_ddpm_cifar10_mi.sh {RATIO} 1.0 1.0',
}

def run(cmd):                         # stream (long steps: finetune/sample)
    print('\n$', cmd, flush=True)
    assert os.system(cmd) == 0, f'FAILED: {cmd}'

def cap(cmd):                         # capture (prune: parse params/macs/overlap)
    print('\n$', cmd, flush=True)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(p.stdout[-1500:])
    if p.returncode != 0:
        print('STDERR', p.stderr[-1500:]); raise SystemExit(f'FAILED: {cmd}')
    return p.stdout

def grab(pat, txt):
    m = re.search(pat, txt); return float(m.group(1)) if m else float('nan')

def metric(cmd, pat, label):         # capture, parse, and SURFACE errors (no silent nan)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    m = re.search(pat, p.stdout)
    if m:
        print(f'{label} = {m.group(1)}')
        return float(m.group(1))
    print(f'!! {label} FAILED (returncode {p.returncode}) -- stderr tail:')
    print((p.stderr or p.stdout)[-800:])
    return float('nan')

def ssim_score():
    return metric('python ddpm_exp/compute_ssim.py '
        '--path run/sample/ddpm_cifar10_pruned/process_0 run/sample/ddpm_cifar10_pretrained/process_0',
        r'ssim:\s*(?:tensor\()?([\-\d.]+)', 'SSIM')

def fid_score():
    return metric('python fid_score.py run/sample/ddpm_cifar10_pruned '
        'run/fid_stats_cifar10.npz --device cuda:0 --batch-size 256',
        r'FID:\s*([\d.]+)', 'FID')

n_samples = FID_SAMPLES if COMPUTE_FID else SSIM_SAMPLES

# FID stats file: auto-build if missing and FID is requested
if COMPUTE_FID and not os.path.exists('run/fid_stats_cifar10.npz'):
    os.makedirs('run', exist_ok=True)
    run('python fid_score.py --save-stats data/cifar10_images '
        'run/fid_stats_cifar10.npz --device cuda:0 --batch-size 256')

# original-model reference for SSIM -- SAME sample count as the pruned runs, so the
# paired SSIM comparison aligns batch-for-batch (mismatched counts crash compute_ssim)
run(f'python ddpm_sample.py --output_dir run/sample/ddpm_cifar10_pretrained '
    f'--batch_size {SAMPLE_BS} --total_samples {n_samples} --ddim_steps {DDIM_STEPS} '
    f'--model_path pretrained/ddpm_ema_cifar10 --skip_type uniform')
results = {}
for name, prune_cmd in METHODS.items():
    print('\n' + '='*60 + f'\n{name}\n' + '='*60, flush=True)
    run('rm -rf run/pruned/ddpm_cifar10_pruned '
        'run/finetuned/ddpm_cifar10_pruned_post_training run/sample/ddpm_cifar10_pruned')
    plog = cap(prune_cmd)             # prune (quiet while it runs; tail printed after)
    run(f'bash scripts/finetune_ddpm_cifar10.sh {ITERS} {WORKERS}')
    run(f'python ddpm_sample.py --output_dir run/sample/ddpm_cifar10_pruned '
        f'--batch_size {SAMPLE_BS} --total_samples {n_samples} --ddim_steps {DDIM_STEPS} '
        f'--pruned_model_ckpt run/finetuned/ddpm_cifar10_pruned_post_training/pruned/unet_ema_pruned.pth '
        f'--model_path run/finetuned/ddpm_cifar10_pruned_post_training --skip_type uniform')
    results[name] = {
        'params'  : grab(r'#Params:.*=>\s*([\d.]+)\s*M', plog),
        'macs'    : grab(r'#MACS:.*=>\s*([\d.]+)\s*G', plog),
        'jaccard' : grab(r'Jaccard\s*=\s*([\d.]+)', plog),
        'spearman': grab(r'Spearman rank corr\s*=\s*([\-\d.]+)', plog),
        'ssim'    : ssim_score(),
        'fid'     : fid_score() if COMPUTE_FID else float('nan'),
    }
    r = results[name]
    print(f"\n>>> {name}: params={r['params']:.2f}M  SSIM={r['ssim']:.4f}"
          + (f"  FID={r['fid']:.2f}" if COMPUTE_FID else ""))



$ python ddpm_sample.py --output_dir run/sample/ddpm_cifar10_pretrained --batch_size 256 --total_samples 10000 --ddim_steps 100 --model_path pretrained/ddpm_ema_cifar10 --skip_type uniform

magnitude

$ rm -rf run/pruned/ddpm_cifar10_pruned run/finetuned/ddpm_cifar10_pruned_post_training run/sample/ddpm_cifar10_pruned

$ bash scripts/prune_ddpm_cifar10.sh 0.3
alse)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
          (conv_shortcut): Conv2d(128, 64, kernel_size=(1, 1), stride=(1, 1))
        )
      )
    )
  )
  (mid_block): UNetMidBlock2D(
    (attentions): ModuleList(
      (0): Attention(
        (group_norm): GroupNorm(32, 160, eps=1e-06, affine=True)
        (to_q): Linear(in_features=160, out_features=179, bias=True)
        (to_k): Linear(in_features=160, out_features=179, bias=True)
        (to_v): Linear(in_features=160, out_features=179, bias=True)
        (to_out): ModuleList(
          (0): Linear(

## Results

In [10]:
print(f'ratio={RATIO}  finetune={ITERS}  ssim_samples={SSIM_SAMPLES}  '
      f'fid={"on" if COMPUTE_FID else "off"}')
print('SSIM higher = better (closer to original) | Jacc/Spear vs magnitude (low = different cuts)\n')
cols = ['params', 'macs', 'ssim', 'jaccard', 'spearman'] + (['fid'] if COMPUTE_FID else [])
print(f"{'method':13s}" + ''.join(f'{c:>9s}' for c in cols))
for name, r in results.items():
    print(f"{name:13s}" + ''.join(f'{r[c]:9.3f}' for c in cols))


ratio=0.3  finetune=1000  ssim_samples=1000  fid=on
SSIM higher = better (closer to original) | Jacc/Spear vs magnitude (low = different cuts)

method          params     macs     ssim  jaccard spearman      fid
magnitude       13.949    2.087    0.595      nan      nan  148.236
mi_combined     13.949    2.087    0.547    0.199    0.036  223.678
